# ERA5 single levels: quickstart

A minimal end-to-end pull of ERA5 single-levels data from the Copernicus
Climate Data Store, opened with xarray, and plotted as a map.

ERA5 is the fifth-generation ECMWF global atmospheric reanalysis. "Single
levels" means surface and near-surface variables (temperature at 2 m, wind
at 10 m, precipitation, radiation, soil moisture) rather than variables on
atmospheric pressure levels. Data runs from 1940 to present at 0.25 degree
resolution, hourly.

Full reference: [`docs/era5-single-levels/README.md`](../docs/era5-single-levels/README.md).

## Before you run this

1. Register for a free CDS account at https://cds.climate.copernicus.eu/
2. Accept the "Licence to use Copernicus Products" for the ERA5 dataset in
   your CDS profile.
3. Copy your Personal Access Token from the CDS profile page and save it to
   `~/.cdsapirc` (see the docs for the exact format).
4. Install the Python stack:
   ```bash
   pip install cdsapi xarray netcdf4 matplotlib cartopy
   ```

The default config below pulls a single hour of 2 metre temperature over the
UK: one NetCDF file, about 30 kB, typically back in under a minute. Edit the
config cell to change variables, dates, or region.

In [ ]:
# ==================================================================
# USER CONFIGURATION - Edit these values for your use case
# ==================================================================
VARIABLES = ["2m_temperature"]
YEARS = ["2023"]
MONTHS = ["07"]
DAYS = ["01"]
HOURS = ["12:00"]
BBOX = [55, -8, 49, 2]  # [north, west, south, east] - UK
OUTPUT_DIR = "../data/era5-single-levels"
OUTPUT_FILENAME = "era5_single_levels_quickstart.nc"
# ==================================================================

## Imports and environment check

Pinning down which versions of the Python stack produced a given figure
saves a lot of time when notebooks behave differently on two machines. The
cell below prints the version of every library we rely on, then calls the
shared credential check so the notebook fails fast with a useful error if
`~/.cdsapirc` is missing.

A small quirk: `cdsapi` does not expose `cdsapi.__version__`, so we use
`importlib.metadata.version()` across the board for consistency.

In [ ]:
import sys
from importlib.metadata import version
from pathlib import Path

import cdsapi
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

print(f"Python       {sys.version.split()[0]}")
for pkg in ["cdsapi", "xarray", "matplotlib", "cartopy", "netcdf4"]:
    print(f"{pkg:12} {version(pkg)}")

# Find the repo root by walking up from CWD until we see CLAUDE.md.
# This works whether Jupyter was launched from the repo root or from
# the notebooks/ directory.
def _find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        if (parent / "CLAUDE.md").exists():
            return parent
    raise RuntimeError(
        "Could not find repo root. Run this notebook from inside the "
        "climate-data-quickstart repository."
    )

repo_root = _find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from common.credentials import check_cds_credentials
check_cds_credentials()
print("\nCDS credentials found.")

## Submit the request

The CDS queues requests server-side. A small single-day request like this
one usually returns in under a minute; larger pulls can take hours. You can
watch the queue position at https://cds.climate.copernicus.eu/requests.

The request dictionary below mirrors what the CDS download form produces if
you tick the equivalent boxes on the website, then click "Show API request".

In [ ]:
output_path = Path(OUTPUT_DIR) / OUTPUT_FILENAME
output_path.parent.mkdir(parents=True, exist_ok=True)

request = {
    "product_type": ["reanalysis"],
    "variable": VARIABLES,
    "year": YEARS,
    "month": MONTHS,
    "day": DAYS,
    "time": HOURS,
    "data_format": "netcdf",
    "download_format": "unarchived",
    "area": BBOX,
}

client = cdsapi.Client()
client.retrieve("reanalysis-era5-single-levels", request).download(str(output_path))
print(f"Saved: {output_path} ({output_path.stat().st_size / 1e6:.2f} MB)")

## Open and inspect

xarray loads the NetCDF lazily and shows the structure: dimensions,
coordinates, and data variables. Three things to notice:

- The 2 metre temperature variable appears as `t2m`, not `2m_temperature`.
  GRIB short names start with a digit (`2t`), and when NetCDF is produced
  from the GRIB archive the names are rewritten to avoid leading digits. The
  full mapping is in the docs.
- A coordinate called `expver` will be present. `expver=1` is final ERA5,
  `expver=5` is preliminary ERA5T. For a request that sits a year or more
  in the past, everything is on `expver=1`.
- Values are in kelvin. We convert to celsius before plotting.

In [ ]:
ds = xr.open_dataset(output_path)
print(ds)

## Plot a map

We use the `PlateCarree` projection (plain latitude-longitude) as a safe
default for mid-latitude regional data; it is fine for the UK and does not
misrepresent scales at this latitude. `RdBu_r` is a diverging colourmap
that reads naturally for temperature. For variables like precipitation or
cloud cover, switch to a sequential colourmap (for example `YlGnBu` or
`viridis`).

The `.squeeze()` call drops the single-element `valid_time` and `expver`
dimensions so we can plot a 2D field directly.

In [ ]:
t2m_var = "t2m" if "t2m" in ds.data_vars else list(ds.data_vars)[0]
t2m = ds[t2m_var].squeeze()
t2m_celsius = t2m - 273.15

fig = plt.figure(figsize=(10, 6))
ax = plt.axes(projection=ccrs.PlateCarree())

t2m_celsius.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap="RdBu_r",
    cbar_kwargs={"label": "2 m temperature (C)"},
)

ax.coastlines(resolution="50m", linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.4, edgecolor="gray")
gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
gl.top_labels = False
gl.right_labels = False

ax.set_title(f"ERA5 2 m temperature, {YEARS[0]}-{MONTHS[0]}-{DAYS[0]} {HOURS[0]} UTC")
plt.tight_layout()
plt.show()

## Next steps

- **Full reference**: [`docs/era5-single-levels/README.md`](../docs/era5-single-levels/README.md)
  for variables, access, licence, citation, and known issues.
- **Full variable list**: [`docs/era5-single-levels/variables.md`](../docs/era5-single-levels/variables.md).
- **Aggregate hourly to daily, monthly, or annual**: xarray's `.resample()`
  handles this directly (for example,
  `ds.resample(valid_time="1D").mean()` for daily means, or `.max()` and
  `.min()` for daily extremes). For very long time series, consider the ERA5
  daily-statistics dataset in this repo (pre-computed) or the
  `-timeseries` sibling dataset for efficient point retrievals.
- **Expand the request**: edit the config cell. Note that large requests
  are queued and heavy ones are deprioritised; it is faster overall to loop
  over smaller chunks (month by month, variable by variable) than to submit
  one monolithic request.
- **Upper-air variables on pressure levels**: see the `era5-pressure-levels`
  entry in this repo.
- **Higher-resolution land-surface data**: see the `era5-land` entry
  (0.1 degree).